<a href="https://colab.research.google.com/github/justinh613/BA305-A1-TeamA2/blob/main/Recipe_Popularity_Prediction_Neural_Net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Predictive Question:** Can you predict the rating of a recipe based on its ingredients?

**Method:** Neural Networks

In [ ]:
# IMPORT PACKAGES

# Data handling
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Splitting and preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer
import re
from collections import Counter

# Deep Learning (Neural Networks)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Models
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, ElasticNet

# Evaluation (regression)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# IMPORT DATASET

url = "https://raw.githubusercontent.com/jalmendral/BA305-A1-TeamA2/main/Data/processed/recipes_clean.csv"
df = pd.read_csv(url)

In [ ]:
# KENNY'S ADDITIONAL CLEANING

# Keep recipes with both ingredient text and an observed rating.
model_df = df.dropna(subset=["ingredients", "avg_rating"]).copy()

# X is the information used to predict.
# y is the value we want to predict.
X = model_df["ingredients"]
y = model_df["avg_rating"]

In [ ]:
# KENNY'S TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
# STRING CLEANING + DUMMY CREATION

import re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Compilation of constants and regular expressions
FRACS = "¼½¾⅐⅑⅒⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞"
SPLIT = re.compile(rf",\s*(?=[\d{FRACS}])")      # commas that start a new ingredient
QTY   = re.compile(rf"^[\d\s/.\-{FRACS}]+")      # leading numbers/fractions
UNITS = re.compile(
    r"^(?:cups?|tablespoons?|teaspoons?|tbsp|tsp|ounces?|oz|pounds?|lbs?|"
    r"pinch(?:es)?|dash(?:es)?|cans?|packages?|pkg|cloves?|sticks?|slices?|"
    r"pints?|quarts?|gallons?)\b\.?\s*(?:of\s+)?"
)
DESCRIPTORS = re.compile(
    r"\b(?:packed|softened|melted|chopped|diced|minced|sliced|grated|shredded|"
    r"large|small|medium|fresh|freshly|ground|divided|beaten|sifted|toasted)\b"
)

def clean_ingredient_item(item):
    if not isinstance(item, str):
        return ""

    # Basic cleaning (lowercasing, parentheticals, post-comma descriptions)
    item = item.lower()
    item = re.sub(r"\(.*?\)", "", item)      # drop parentheticals like "(8 ounce)"
    item = item.split(",")[0]                # drop prep notes after a comma

    # Remove measurements and units
    item = QTY.sub("", item).strip()
    item = UNITS.sub("", item)
    item = DESCRIPTORS.sub("", item)

    # Refined Cleaning: Remove leading symbols (hyphens, percents, bullet points, etc.), 'or', and leftover digits
    item = re.sub(r"^(?:%|or|\d%|\d|\s|[-#*+])+", "", item).strip()

    # Normalize white spaces
    item = re.sub(r"\s+", " ", item).strip()

    # Discard if too short or if it contains only symbols/numbers
    if len(item) < 2 or re.match(r"^[\d\W_]+$", item):
        return ""

    return item

def parse_ingredients_comprehensive(s):
    if not isinstance(s, str):
        return []
    items = [clean_ingredient_item(x) for x in SPLIT.split(s)]
    return sorted({i for i in items if i})   # unique, non-empty

# Apply the combined cleaning pipeline to datasets
df["ing_list"] = df["ingredients"].apply(parse_ingredients_comprehensive)
model_df["ing_list"] = model_df["ingredients"].apply(parse_ingredients_comprehensive)

# Generate dummy variables
mlb_comprehensive = MultiLabelBinarizer()
ingredients_encoded_comp = mlb_comprehensive.fit_transform(model_df['ing_list'])
raw_ingredients_df = pd.DataFrame(ingredients_encoded_comp, columns=mlb_comprehensive.classes_, index=model_df.index)

# Filter out columns starting with leading symbols or non-word characters for safety
clean_cols = [col for col in raw_ingredients_df.columns if not re.match(r'^[\W_]', col)]
ingredients_df = raw_ingredients_df[clean_cols]

In [ ]:
# CONSOLIDATED DYNAMIC INGREDIENT GROUPING & FREQUENCY FILTERING
import re
from collections import Counter
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Dynamically discover common head words (the last word of each unique ingredient)
all_unique_ingredients = list(ingredients_df.columns)
head_words = [ing.strip().split()[-1] for ing in all_unique_ingredients if ing.strip().split()]

# Find head words appearing at least 5 times in the unique ingredient vocabulary
head_word_counts = Counter(head_words)
DYNAMIC_LIMIT = 5
common_head_words = [word for word, count in head_word_counts.items() if count >= DYNAMIC_LIMIT and len(word) > 2]

# Helper to map complex ingredients to our discovered base categories
def dynamic_group_ingredient(item):
    for keyword in common_head_words:
        if f" {keyword}" in f" {item}":
            return keyword
    return item

# 2. Apply the dynamic grouping map to the ingredient lists
model_df["dynamic_grouped_ing_list"] = model_df["ing_list"].apply(
    lambda ing_list: sorted({dynamic_group_ingredient(ing) for ing in ing_list})
)

# 3. Create dummy variables from the grouped ingredients
mlb_dynamic = MultiLabelBinarizer()
ingredients_encoded_dynamic = mlb_dynamic.fit_transform(model_df['dynamic_grouped_ing_list'])
raw_ingredients_dynamic_df = pd.DataFrame(
    ingredients_encoded_dynamic,
    columns=mlb_dynamic.classes_,
    index=model_df.index
)

# Remove any remaining columns that begin with non-word characters for safety
clean_dynamic_cols = [col for col in raw_ingredients_dynamic_df.columns if not re.match(r'^[^\w]', col)]
dynamic_ingredients_df = raw_ingredients_dynamic_df[clean_dynamic_cols]

# 4. Filter out any ingredient appearing in fewer than 10 recipes total
dynamic_frequencies = dynamic_ingredients_df.sum(axis=0)
filtered_dynamic_ingredients_df = dynamic_ingredients_df[dynamic_frequencies[dynamic_frequencies >= 10].index]

# 5. Output summary
print("=== CONSOLIDATION SUMMARY ===")
print(f"Original vocabulary size: {ingredients_df.shape[1]} columns")
print(f"Dynamically discovered base categories: {len(common_head_words)}")
print(f"Final simplified features (Grouped & appearing >= 10 times): {filtered_dynamic_ingredients_df.shape[1]} columns")
print("\nTop 15 most frequent consolidated ingredients:")
display(dynamic_frequencies.loc[filtered_dynamic_ingredients_df.columns].sort_values(ascending=False).head(15))

=== CONSOLIDATION SUMMARY ===
Original vocabulary size: 11403 columns
Dynamically discovered base categories: 402
Final simplified features (Grouped & appearing >= 10 times): 369 columns

Top 15 most frequent consolidated ingredients:


,0
salt,6409
sugar,5783
butter,5111
oil,5045
onion,4874
pepper,4596
flour,4434
garlic,3733
cheese,3709
powder,3473


Note - all cells up to this point are cleaning the data and using the same methods as established in the ridge/lasso notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

# 1. Prepare features and targets based on the train/test split index alignment
# Extract clean features using the matching indices from the split
X_train_feat = filtered_dynamic_ingredients_df.loc[X_train.index].values.astype(np.float32)
X_test_feat = filtered_dynamic_ingredients_df.loc[X_test.index].values.astype(np.float32)

y_train_arr = y_train.values.astype(np.float32).reshape(-1, 1)
y_test_arr = y_test.values.astype(np.float32).reshape(-1, 1)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_feat)
y_train_tensor = torch.tensor(y_train_arr)
X_test_tensor = torch.tensor(X_test_feat)
y_test_tensor = torch.tensor(y_test_arr)

# 2. Define the Neural Network Architecture
class RecipeRatingNN(nn.Module):
    def __init__(self, input_dim):
        super(RecipeRatingNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 4),
            nn.Tanh(),
            nn.Linear(4, 4),
            nn.Tanh(),
            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.network(x)

# 3. Setup 5-fold cross-validation on learning rates
lrs = [0.001, 0.01, 0.05, 0.1]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
epochs = 30
batch_size = 64

input_dim = X_train_feat.shape[1]
best_lr = None
best_val_loss = float('inf')
results = {}

print("--- Starting 5-Fold Cross-Validation on Learning Rate ---")
for lr in lrs:
    fold_losses = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
        # Subsets
        train_sub_x, train_sub_y = X_train_tensor[train_idx], y_train_tensor[train_idx]
        val_sub_x, val_sub_y = X_train_tensor[val_idx], y_train_tensor[val_idx]

        train_loader = DataLoader(TensorDataset(train_sub_x, train_sub_y), batch_size=batch_size, shuffle=True)

        # Initialize Model, Optimizer, Loss Function
        model = RecipeRatingNN(input_dim)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Training loop for fold
        model.train()
        for epoch in range(epochs):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                predictions = model(batch_x)
                loss = criterion(predictions, batch_y)
                loss.backward()
                optimizer.step()

        # Evaluation on validation fold
        model.eval()
        with torch.no_grad():
            val_preds = model(val_sub_x)
            val_loss = criterion(val_preds, val_sub_y).item()
            fold_losses.append(val_loss)

    mean_cv_loss = np.mean(fold_losses)
    results[lr] = mean_cv_loss
    print(f"LR: {lr} | Mean CV MSE: {mean_cv_loss:.5f}")

    if mean_cv_loss < best_val_loss:
        best_val_loss = mean_cv_loss
        best_lr = lr

print(f"\nBest Learning Rate: {best_lr} (Mean CV MSE: {best_val_loss:.5f})")

--- Starting 5-Fold Cross-Validation on Learning Rate ---
LR: 0.001 | Mean CV MSE: 0.16927
LR: 0.01 | Mean CV MSE: 0.18617
LR: 0.05 | Mean CV MSE: 0.17170
LR: 0.1 | Mean CV MSE: 0.17619

Best Learning Rate: 0.001 (Mean CV MSE: 0.16927)


In [ ]:
# 4. Train final model with the best learning rate and evaluate on test set
final_model = RecipeRatingNN(input_dim)
optimizer = optim.Adam(final_model.parameters(), lr=best_lr)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)

final_model.train()
for epoch in range(50):  # Run a slightly longer final training
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(final_model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

# Evaluate on Test Set
final_model.eval()
with torch.no_grad():
    test_predictions = final_model(X_test_tensor).numpy()

from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test_arr, test_predictions)
r2 = r2_score(y_test_arr, test_predictions)

print("=== Final Evaluation on Test Set ===")
print(f"Test MSE: {mse:.5f}")
print(f"Test R² Score: {r2:.5f}")

=== Final Evaluation on Test Set ===
Test MSE: 0.17552
Test R² Score: -0.02087


**First Round Analysis**

This neural net was extremely constrained (2 hidden layers, 4 nodes) so it was not tuned to the patterns in the data. We will most likely need to introduce more complexity to get an accurate regression reading.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import numpy as np

# Input dimension is 369
input_dim = X_train_feat.shape[1]

# Square root heuristic: sqrt(input_dim * output_dim) ≈ 19 nodes.
# We will test 3, 4, and 5 hidden layers starting near this baseline and tapering down:
architectures = {
    "3_layers": [19, 12, 6],
    "4_layers": [19, 14, 10, 6],
    "5_layers": [19, 15, 12, 8, 4]
}

# Dynamic Neural Network Builder using LeakyReLU
class LeakyRecipeNN(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super(LeakyRecipeNN, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.01))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Settings for evaluation
lr = 0.001
kf = KFold(n_splits=3, shuffle=True, random_state=42)
epochs = 20
batch_size = 64

arch_results = {}
best_arch_name = None
best_arch_loss = float('inf')

print("--- Starting Grid Search (LeakyReLU & Square Root Heuristic) ---")
for name, hidden_dims in architectures.items():
    fold_losses = []
    print(f"Evaluating {name}: {hidden_dims}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
        train_sub_x, train_sub_y = X_train_tensor[train_idx], y_train_tensor[train_idx]
        val_sub_x, val_sub_y = X_train_tensor[val_idx], y_train_tensor[val_idx]

        train_loader = DataLoader(TensorDataset(train_sub_x, train_sub_y), batch_size=batch_size, shuffle=True)

        model = LeakyRecipeNN(input_dim, hidden_dims)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for epoch in range(epochs):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(val_sub_x)
            val_loss = criterion(val_preds, val_sub_y).item()
            fold_losses.append(val_loss)

    mean_cv_loss = np.mean(fold_losses)
    arch_results[name] = mean_cv_loss
    print(f"{name} Mean CV MSE: {mean_cv_loss:.5f}\n")

    if mean_cv_loss < best_arch_loss:
        best_arch_loss = mean_cv_loss
        best_arch_name = name

print(f"Best Architecture: {best_arch_name} ({architectures[best_arch_name]}) with Mean CV MSE: {best_arch_loss:.5f}")

--- Starting Grid Search (LeakyReLU & Square Root Heuristic) ---
Evaluating 3_layers: [19, 12, 6]
3_layers Mean CV MSE: 0.17448

Evaluating 4_layers: [19, 14, 10, 6]
4_layers Mean CV MSE: 0.18160

Evaluating 5_layers: [19, 15, 12, 8, 4]
5_layers Mean CV MSE: 0.17543

Best Architecture: 3_layers ([19, 12, 6]) with Mean CV MSE: 0.17448


In [ ]:
# Train and evaluate the best LeakyReLU architecture on the full test set
best_hidden_dims = architectures[best_arch_name]
final_tuned_model = LeakyRecipeNN(input_dim, best_hidden_dims)
optimizer = optim.Adam(final_tuned_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)

final_tuned_model.train()
for epoch in range(40):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(final_tuned_model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

final_tuned_model.eval()
with torch.no_grad():
    tuned_test_preds = final_tuned_model(X_test_tensor).numpy()

tuned_mse = mean_squared_error(y_test_arr, tuned_test_preds)
tuned_r2 = r2_score(y_test_arr, tuned_test_preds)

print("=== Tuned LeakyReLU Architecture Evaluation on Test Set ===")
print(f"Best Hidden Architecture: {best_hidden_dims}")
print(f"Test MSE: {tuned_mse:.5f}")
print(f"Test R² Score: {tuned_r2:.5f}")

=== Tuned LeakyReLU Architecture Evaluation on Test Set ===
Best Hidden Architecture: [19, 12, 6]
Test MSE: 0.19892
Test R² Score: -0.15695


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Prepare features and targets based on the train/test split index alignment
X_train_feat = filtered_dynamic_ingredients_df.loc[X_train.index].values.astype(np.float32)
X_test_feat = filtered_dynamic_ingredients_df.loc[X_test.index].values.astype(np.float32)

y_train_arr = y_train.values.astype(np.float32).reshape(-1, 1)
y_test_arr = y_test.values.astype(np.float32).reshape(-1, 1)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_feat)
y_train_tensor = torch.tensor(y_train_arr)
X_test_tensor = torch.tensor(X_test_feat)
y_test_tensor = torch.tensor(y_test_arr)

# Input dimension based on filtered features
input_dim = X_train_feat.shape[1]

# Define architectural candidates for grid search
architectures = {
    '2_layers_wide': [64, 16],
    '3_layers_medium': [32, 16, 8],
    '3_layers_deep': [64, 32, 16],
    '4_layers_tapered': [64, 32, 16, 8],
    '5_layers_deep': [128, 64, 32, 16, 8]
}

# Reusable neural network module
class DynamicRecipeNN(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super(DynamicRecipeNN, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.01))
            layers.append(nn.Dropout(p=0.2))  # Add dropout to mitigate overfitting
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Grid Search settings
lr = 0.001
kf = KFold(n_splits=3, shuffle=True, random_state=42)
epochs = 25
batch_size = 64

arch_results = {}
best_arch_name = None
best_arch_loss = float('inf')

print("--- Starting Expanded Architecture Grid Search ---")
for name, hidden_dims in architectures.items():
    fold_losses = []
    print(f"Evaluating {name}: {hidden_dims}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):
        train_sub_x, train_sub_y = X_train_tensor[train_idx], y_train_tensor[train_idx]
        val_sub_x, val_sub_y = X_train_tensor[val_idx], y_train_tensor[val_idx]

        train_loader = DataLoader(TensorDataset(train_sub_x, train_sub_y), batch_size=batch_size, shuffle=True)

        model = DynamicRecipeNN(input_dim, hidden_dims)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)  # L2 regularization

        model.train()
        for epoch in range(epochs):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(val_sub_x)
            val_loss = criterion(val_preds, val_sub_y).item()
            fold_losses.append(val_loss)

    mean_cv_loss = np.mean(fold_losses)
    arch_results[name] = mean_cv_loss
    print(f"{name} Mean CV MSE: {mean_cv_loss:.5f}\n")

    if mean_cv_loss < best_arch_loss:
        best_arch_loss = mean_cv_loss
        best_arch_name = name

print(f"Best Architecture: {best_arch_name} ({architectures[best_arch_name]}) with Mean CV MSE: {best_arch_loss:.5f}")

--- Starting Expanded Architecture Grid Search ---
Evaluating 2_layers_wide: [64, 16]
2_layers_wide Mean CV MSE: 0.18948

Evaluating 3_layers_medium: [32, 16, 8]
3_layers_medium Mean CV MSE: 0.19544

Evaluating 3_layers_deep: [64, 32, 16]
3_layers_deep Mean CV MSE: 0.19283

Evaluating 4_layers_tapered: [64, 32, 16, 8]
4_layers_tapered Mean CV MSE: 0.18745

Evaluating 5_layers_deep: [128, 64, 32, 16, 8]
5_layers_deep Mean CV MSE: 0.18967

Best Architecture: 4_layers_tapered ([64, 32, 16, 8]) with Mean CV MSE: 0.18745


In [ ]:
# Ensure all required variables and classes are accessible
best_hidden_dims = architectures[best_arch_name]
final_search_model = DynamicRecipeNN(input_dim, best_hidden_dims)
optimizer = optim.Adam(final_search_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)

final_search_model.train()
for epoch in range(40):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(final_search_model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

# Evaluate on Test Set
final_search_model.eval()
with torch.no_grad():
    final_test_preds = final_search_model(X_test_tensor).numpy()

final_test_mse = mean_squared_error(y_test_arr, final_test_preds)
final_test_r2 = r2_score(y_test_arr, final_test_preds)

print("=== Final Evaluation on Test Set ===")
print(f"Best Architecture: {best_hidden_dims}")
print(f"Test MSE: {final_test_mse:.5f}")
print(f"Test R² Score: {final_test_r2:.5f}")

=== Final Evaluation on Test Set ===
Best Architecture: [64, 32, 16, 8]
Test MSE: 0.16914
Test R² Score: 0.01626


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. Prepare the complete dataset
X_all_feat = filtered_dynamic_ingredients_df.values.astype(np.float32)
y_all_arr = y.values.astype(np.float32).reshape(-1, 1)

X_all_tensor = torch.tensor(X_all_feat)
y_all_tensor = torch.tensor(y_all_arr)

# 2. Instantiate the optimal architecture: [64, 32, 16, 8]
optimal_dims = [64, 32, 16, 8]
optimal_model = DynamicRecipeNN(input_dim, optimal_dims)
optimizer = optim.Adam(optimal_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.MSELoss()

# 3. Fit on the entire dataset
full_loader = DataLoader(TensorDataset(X_all_tensor, y_all_tensor), batch_size=64, shuffle=True)

optimal_model.train()
for epoch in range(40):
    for batch_x, batch_y in full_loader:
        optimizer.zero_grad()
        loss = criterion(optimal_model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

# 4. Evaluate on the entire dataset
optimal_model.eval()
with torch.no_grad():
    all_predictions = optimal_model(X_all_tensor).numpy()

overall_mse = mean_squared_error(y_all_arr, all_predictions)
print("=== Optimal Model Trained & Evaluated on the Entire Dataset ===")
print(f"Overall Dataset MSE: {overall_mse:.5f}")

=== Optimal Model Trained & Evaluated on the Entire Dataset ===
Overall Dataset MSE: 0.13770
